In [2]:
from datetime import date

import pandas as pd
import yaml

Para inicializar la dimension creamos un dataframe con las 24 horas y 60 minutos del dia (granularidad por minuto).

In [3]:
dim_hora = pd.DataFrame({
    "minute_of_day": range(24 * 60)
})

dim_hora["hour"] = dim_hora["minute_of_day"] // 60
dim_hora["minute"] = dim_hora["minute_of_day"] % 60

dim_hora.head()

,minute_of_day,hour,minute
0,0,0,0
1,1,0,1
2,2,0,2
3,3,0,3
4,4,0,4


Vamos a agregar columnas utiles para analisis por minuto y por hora

In [4]:
dim_hora["hour_12"] = dim_hora["hour"].apply(lambda h: 12 if h % 12 == 0 else h % 12)
dim_hora["meridiem"] = dim_hora["hour"].apply(lambda h: "AM" if h < 12 else "PM")
dim_hora["time_str"] = dim_hora.apply(lambda r: f"{r['hour']:02d}:{r['minute']:02d}:00", axis=1)
dim_hora["time_12_str"] = dim_hora.apply(
    lambda r: f"{r['hour_12']:02d}:{r['minute']:02d} {r['meridiem']}", axis=1
)

dim_hora.head()

,minute_of_day,hour,minute,hour_12,meridiem,time_str,time_12_str
0,0,0,0,12,AM,00:00:00,12:00 AM
1,1,0,1,12,AM,00:01:00,12:01 AM
2,2,0,2,12,AM,00:02:00,12:02 AM
3,3,0,3,12,AM,00:03:00,12:03 AM
4,4,0,4,12,AM,00:04:00,12:04 AM


Clasificamos cada minuto en una franja del dia

In [5]:
# def franja_horaria(hour):
#     if 0 <= hour < 6:
#         return "madrugada"
#     if 6 <= hour < 12:
#         return "manana"
#     if 12 <= hour < 18:
#         return "tarde"
#     return "noche"

# dim_hora["day_period"] = dim_hora["hour"].apply(franja_horaria)
# dim_hora["is_business_hour"] = dim_hora["hour"].between(8, 17)
# dim_hora["saved"] = date.today()

dim_hora.head()

,minute_of_day,hour,minute,hour_12,meridiem,time_str,time_12_str
0,0,0,0,12,AM,00:00:00,12:00 AM
1,1,0,1,12,AM,00:01:00,12:01 AM
2,2,0,2,12,AM,00:02:00,12:02 AM
3,3,0,3,12,AM,00:03:00,12:03 AM
4,4,0,4,12,AM,00:04:00,12:04 AM


In [6]:
from sqlalchemy import create_engine

with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_etl = config['ETL_PRO']

url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
etl_conn = create_engine(url_etl)

In [7]:
dim_hora.to_sql('dim_hora', etl_conn, if_exists='replace', index_label='key_dim_hora')

440